# Face model v2 — accuracy experiments (Colab GPU)
Baseline = v1 (EfficientNet-B0, res 96, 20 ep, CutMix off, class-weight power 0.5): **val Macro-F1 0.6566**, val acc 0.660.
Every run changes ONE thing, same seed and the same validation split. Test data is never touched here (finalize.py does that once, locally).

**Before running** put 2 files in Google Drive folder `MyDrive/face-v2/`:
- `Face-Lab-code.zip` (from `Face-Lab/colab/`)
- `archive (4).zip` (the FER2013 zip; or rename to `fer2013.zip`)
Runtime → Change runtime type → GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
print(os.listdir(D))
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
for s in ('train','test'):
    print(s, sum(len(files) for _,_,files in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab

## 2 — Experiments (one change each)

In [ ]:
EXPS = {
  'v2_mix05':   '--mix 0.5',                       # CutMix on
  'v2_cw075':   '--cw-power 0.75',                 # push sad / fear (rarer classes get more weight)
  'v2_res112':  '--res 112',                       # more pixels (costs latency)
  'v2_mix_e30': '--mix 0.5 --epochs 30 --patience 8',
}
BASE = '--arch efficientnet_b0 --res 96 --epochs 20 --head-epochs 1 --unfreeze 0.0 --seed 42'
import subprocess
for tag, extra in EXPS.items():
    cmd = f'python train.py {BASE} {extra} --tag {tag}'      # later flags override BASE
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-600:], r.stderr[-400:] if r.returncode else '', flush=True)

## 3 — Compare with v1 (validation set)

In [ ]:
import json, glob
V1 = 0.6566
rows=[]
for f in sorted(glob.glob('results/v2_*.json')):
    r=json.load(open(f)); pc=r['val']['per_class']
    rows.append((r['tag'], r['best_val_macro_f1'], r['val']['accuracy'], pc['sad']['f1'], pc['fear']['f1'], r['best_epoch'], r['train_minutes']))
print(f"{'run':12s} {'macroF1':>8s} {'acc':>6s} {'sad':>6s} {'fear':>6s} {'ep':>3s} {'min':>5s}   (v1 macroF1 {V1}, sad .512, fear .477)")
for t,m,a,s,fe,ep,mn in sorted(rows,key=lambda x:-x[1]):
    print(f"{t:12s} {m:8.4f} {a:6.3f} {s:6.3f} {fe:6.3f} {ep:3d} {mn:5.1f}  {'+' if m>V1 else '-'}{abs(m-V1):.4f}")

## 4 — Save results back to Drive

In [ ]:
import shutil, os
OUT='/content/drive/MyDrive/face-v2/out'
os.makedirs(OUT, exist_ok=True)
shutil.make_archive('/content/face_v2_results','zip','/content/Main_demo/Face-Lab','results')   # json logs of every run
shutil.copy('/content/face_v2_results.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.pt'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True)
        shutil.copy(f'runs/{t}/best.pt', f'{OUT}/{t}/best.pt')
print(os.listdir(OUT))